# AI Text Detector — BERT Fine-tuning
Run this notebook in **Google Colab** with a GPU runtime:  
`Runtime → Change runtime type → T4 GPU`

## 1. Install dependencies

In [18]:
!pip install -q transformers scikit-learn pandas torch

In [19]:
import pandas as pd
import torch
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import clip_grad_norm_
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import RobertaTokenizer, RobertaForSequenceClassification

import shutil
from google.colab import files
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np

## 2. Mount Google Drive & load dataset
1. Upload `arxiv_pairs_1000.csv` (the **raw pairs** file, not the flat dataset) to your Google Drive
2. Run this cell — it will ask you to sign in and grant access

> **Why the raw pairs file?** We need both `real_abstract` and `ai_abstract` columns together so we can split by paper before flattening. The processed `arxiv_dataset_1000.csv` loses that pairing info.

In [20]:
from google.colab import drive
drive.mount("/content/drive")

# Path to the RAW pairs CSV (has real_abstract + ai_abstract columns per paper)
DATASET_PATH = "/content/drive/MyDrive/textdetector/arxiv_pairs_1000.csv"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Imports & GPU check

In [21]:


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA A100-SXM4-40GB


## 4. Load & preprocess data

In [22]:
def convert_to_classification(pairs_df):
    """Flatten a pairs DataFrame into individual (text, label) rows."""
    human_df = pd.DataFrame({"text": pairs_df["real_abstract"].values, "label": 0})
    ai_df    = pd.DataFrame({"text": pairs_df["ai_abstract"].values,   "label": 1})
    return pd.concat([human_df, ai_df], ignore_index=True)


def split_and_flatten(pairs_df):
    """
    Split by PAPER (pair) first, then flatten into (text, label) rows.

    Previously the code flattened first and split after, meaning the human
    abstract of paper X could land in train while the AI abstract of the same
    paper landed in test. The model then learned paper-specific content during
    training and exploited it at test time — inflating accuracy artificially.

    Splitting at the pair level guarantees no paper's content appears in more
    than one split.
    """
    train_pairs, temp_pairs = train_test_split(pairs_df, test_size=0.2, random_state=42)
    val_pairs,   test_pairs = train_test_split(temp_pairs, test_size=0.5, random_state=42)

    train_df = convert_to_classification(train_pairs)
    val_df   = convert_to_classification(val_pairs)
    test_df  = convert_to_classification(test_pairs)

    print(f"Papers — Train: {len(train_pairs)} | Val: {len(val_pairs)} | Test: {len(test_pairs)}")
    print(f"Rows   — Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    return train_df, val_df, test_df


pairs_df = pd.read_csv(DATASET_PATH)
pairs_df = pairs_df.dropna(subset=["real_abstract", "ai_abstract"])
print(f"Papers after dedup: {len(pairs_df)}")

train_df, val_df, test_df = split_and_flatten(pairs_df)

Papers after dedup: 981
Papers — Train: 784 | Val: 98 | Test: 99
Rows   — Train: 1568 | Val: 196 | Test: 198


## 5. Dataset class & DataLoaders

In [23]:
class EssayDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=256):
        self.texts     = dataframe["text"].tolist()
        self.labels    = dataframe["label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long)
        }

tokenizer    = BertTokenizer.from_pretrained("bert-base-uncased")
train_loader = DataLoader(EssayDataset(train_df, tokenizer), batch_size=16, shuffle=True)
val_loader   = DataLoader(EssayDataset(val_df,   tokenizer), batch_size=16, shuffle=False)
test_loader  = DataLoader(EssayDataset(test_df,  tokenizer), batch_size=16, shuffle=False)
print("DataLoaders ready.")

DataLoaders ready.


## 6. Training loop

In [24]:

def train_model(model, train_loader, val_loader, optimizer, device, epochs=3):
    history = {"train_loss": [], "val_acc": [], "val_loss": []}

    for epoch in range(1, epochs + 1):
        # training
        model.train()
        total_train_loss = 0
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            outputs.loss.backward()
            clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_train_loss += outputs.loss.item()

        avg_train_loss = total_train_loss / len(train_loader)

        # validation
        model.eval()
        total_val_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["label"].to(device)

                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                total_val_loss += outputs.loss.item()

                preds = outputs.logits.argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        avg_val_loss = total_val_loss / len(val_loader)
        val_acc = correct / total

        # history
        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch}/{epochs} | Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")

    return history

## 7. Evaluation loop

In [25]:
def evaluate_model(model, test_loader, device, return_probs=True):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            preds = logits.argmax(dim=1)
            probs = torch.softmax(logits, dim=1)[:,1]

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    # getting metrics
    accuracy = sum([p==l for p,l in zip(all_preds, all_labels)]) / len(all_labels)
    class_report = classification_report(all_labels, all_preds, digits=4)
    conf_matrix = confusion_matrix(all_labels, all_preds)
    roc_auc = roc_auc_score(all_labels, all_probs) if return_probs else None

    print(f"Test Accuracy: {accuracy:.4f}")
    print("Classification Report:\n", class_report)
    print("Confusion Matrix:\n", conf_matrix)
    if roc_auc is not None:
        print(f"AUC-ROC: {roc_auc:.4f}")


# Train and test BERT model

In [26]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2).to(device)
optimizer = AdamW(bert_model.parameters(), lr=2e-5)

train_model(bert_model, train_loader, val_loader, optimizer, device, epochs=3)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/3 | Train Loss: 0.4981 | Val Loss: 0.2089 | Val Acc: 0.8980
Epoch 2/3 | Train Loss: 0.2214 | Val Loss: 0.4875 | Val Acc: 0.8724
Epoch 3/3 | Train Loss: 0.1563 | Val Loss: 0.4366 | Val Acc: 0.8878


{'train_loss': [0.4981049590420966, 0.221356883264926, 0.1563426878740441],
 'val_acc': [0.8979591836734694, 0.8724489795918368, 0.8877551020408163],
 'val_loss': [0.20888593363074157, 0.4875317827726786, 0.4365884904982522]}

In [27]:
print(evaluate_model(bert_model, test_loader, device, return_probs=True))

Test Accuracy: 0.8485
Classification Report:
               precision    recall  f1-score   support

           0     1.0000    0.6970    0.8214        99
           1     0.7674    1.0000    0.8684        99

    accuracy                         0.8485       198
   macro avg     0.8837    0.8485    0.8449       198
weighted avg     0.8837    0.8485    0.8449       198

Confusion Matrix:
 [[69 30]
 [ 0 99]]
AUC-ROC: 0.9784
None


In [28]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=10000, sublinear_tf=True)
X_train = tfidf.fit_transform(train_df["text"])
X_test  = tfidf.transform(test_df["text"])

lr = LogisticRegression(max_iter=1000, C=1.0)
lr.fit(X_train, train_df["label"])
lr_preds = lr.predict(X_test)
lr_acc = accuracy_score(test_df["label"], lr_preds)
print(f"TF-IDF + LogReg accuracy: {lr_acc:.4f}")
print()
if lr_acc > 0.85:
    print("A bag-of-words model nearly matches BERT — the signal is in")
    print("surface-level word choice, not deep semantic understanding.")
else:
    print("BERT outperforms the baseline — it's learning something beyond word frequencies.")

feature_names = tfidf.get_feature_names_out()
coefs = lr.coef_[0]

top_n = 20
ai_idx    = np.argsort(coefs)[-top_n:][::-1]
human_idx = np.argsort(coefs)[:top_n]

print("\n── Top phrases predicting AI (label=1) ──")
for i in ai_idx:
    print(f"  {feature_names[i]:<35} coef={coefs[i]:+.3f}")

print("\n── Top phrases predicting HUMAN (label=0) ──")
for i in human_idx:
    print(f"  {feature_names[i]:<35} coef={coefs[i]:+.3f}")

bert_model.eval()
all_texts, all_labels_raw, all_preds_raw, all_probs_raw = [], [], [], []

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
        probs   = torch.softmax(outputs.logits, dim=1)[:, 1]
        preds   = outputs.logits.argmax(dim=1)

        start = batch_idx * test_loader.batch_size
        end   = start + labels.size(0)
        all_texts.extend(test_df["text"].iloc[start:end].tolist())
        all_labels_raw.extend(labels.cpu().numpy())
        all_preds_raw.extend(preds.cpu().numpy())
        all_probs_raw.extend(probs.cpu().numpy())

results_df = pd.DataFrame({
    "text":       all_texts,
    "label":      all_labels_raw,
    "pred":       all_preds_raw,
    "ai_prob":    all_probs_raw,
})
results_df["correct"] = results_df["label"] == results_df["pred"]
results_df["label_str"] = results_df["label"].map({0: "human", 1: "ai"})
results_df["pred_str"]  = results_df["pred"].map({0: "human", 1: "ai"})

errors = results_df[~results_df["correct"]].copy()
errors["confidence"] = errors["ai_prob"].apply(lambda p: max(p, 1-p))
errors = errors.sort_values("confidence", ascending=False)

print(f"\n── Errors: {len(errors)} / {len(results_df)} misclassified ──")
print(f"   False positives (human→AI): {((errors.label==0)&(errors.pred==1)).sum()}")
print(f"   False negatives (AI→human): {((errors.label==1)&(errors.pred==0)).sum()}")

TF-IDF + LogReg accuracy: 0.9444

A bag-of-words model nearly matches BERT — the signal is in
surface-level word choice, not deep semantic understanding.

── Top phrases predicting AI (label=1) ──
  novel                               coef=+3.076
  robust                              coef=+2.391
  critical                            coef=+1.948
  diverse                             coef=+1.787
  often                               coef=+1.780
  significantly                       coef=+1.774
  significant                         coef=+1.748
  this                                coef=+1.625
  crucial                             coef=+1.495
  superior                            coef=+1.493
  its                                 coef=+1.488
  comprehensive                       coef=+1.360
  consistently                        coef=+1.297
  findings                            coef=+1.280
  substantial                         coef=+1.239
  particularly                        coef=+1.238
  l

## 8. Diagnosis — why is accuracy so high?

Three checks:
1. **TF-IDF baseline** — if a bag-of-words logistic regression matches BERT, the signal is trivial surface-level features, not deep semantics
2. **Top discriminating n-grams** — which specific words/phrases give it away
3. **Error analysis** — inspect what the model gets wrong to understand the decision boundary

# 9. Train and test RoBERTa model

In [29]:
roberta_tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# Rebuild DataLoaders with the RoBERTa tokenizer — BERT and RoBERTa use different
# vocabularies and special tokens, so we can't reuse the loaders from cell 10.
roberta_train_loader = DataLoader(EssayDataset(train_df, roberta_tokenizer), batch_size=16, shuffle=True)
roberta_val_loader   = DataLoader(EssayDataset(val_df,   roberta_tokenizer), batch_size=16, shuffle=False)
roberta_test_loader  = DataLoader(EssayDataset(test_df,  roberta_tokenizer), batch_size=16, shuffle=False)

roberta_model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2).to(device)
optimizer = torch.optim.AdamW(roberta_model.parameters(), lr=2e-5)

train_model(roberta_model, roberta_train_loader, roberta_val_loader, optimizer, device, epochs=3)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/3 | Train Loss: 0.4858 | Val Loss: 0.2890 | Val Acc: 0.8776
Epoch 2/3 | Train Loss: 0.3074 | Val Loss: 0.3260 | Val Acc: 0.9235
Epoch 3/3 | Train Loss: 0.1677 | Val Loss: 0.3296 | Val Acc: 0.9184


{'train_loss': [0.4858282100105164, 0.3074013020044991, 0.16767426985268463],
 'val_acc': [0.8775510204081632, 0.923469387755102, 0.9183673469387755],
 'val_loss': [0.2889580097622596, 0.3260342293204023, 0.32956070083981526]}

In [30]:
print(evaluate_model(roberta_model, roberta_test_loader, device, return_probs=True))

Test Accuracy: 0.9444
Classification Report:
               precision    recall  f1-score   support

           0     0.9400    0.9495    0.9447        99
           1     0.9490    0.9394    0.9442        99

    accuracy                         0.9444       198
   macro avg     0.9445    0.9444    0.9444       198
weighted avg     0.9445    0.9444    0.9444       198

Confusion Matrix:
 [[94  5]
 [ 6 93]]
AUC-ROC: 0.9904
None


## 9. Save the model (optional)
This saves to Colab's temporary storage. Download it before the session ends.

In [31]:
bert_model.save_pretrained("bert_textdetector")
tokenizer.save_pretrained("bert_textdetector")
print("Model saved to ./bert_textdetector/")

shutil.make_archive("bert_textdetector", "zip", "bert_textdetector")
files.download("bert_textdetector.zip")

Model saved to ./bert_textdetector/


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>